# Malware Detection Using Machine Learning

This notebook applies multiple ML algorithms (Random Forest, Decision Tree, Logistic Regression, SVM, KNN) to a PE-file malware dataset and compares their performance.

**Dataset:** [PE Files Malwares](https://www.kaggle.com/datasets/maidaly/pe-files-malwares) from Kaggle

## 1. Setup & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
plt.style.use('ggplot')

## 2. Load Dataset

Upload `dataset_malwares.csv` when prompted, or mount Google Drive / use Kaggle API.

In [ ]:
# Option A: Upload file directly in Colab
# from google.colab import files
# uploaded = files.upload()

# Option B: If using Kaggle on Colab
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d maidaly/pe-files-malwares -p . --unzip

# Option C: Direct path (Kaggle notebook)
# data = pd.read_csv('/kaggle/input/pe-files-malwares/dataset_malwares.csv')

data = pd.read_csv('dataset_malwares.csv')
data.head()

In [ ]:
data.info()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.countplot(x=data['Malware'])
ax.set_xticklabels(['Benign', 'Malware'])
plt.show()

## 4. Data Preparation

In [ ]:
used_data = data.drop(['Name', 'Machine', 'TimeDateStamp', 'Malware'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(used_data, data['Malware'], test_size=0.2, random_state=0)
print(f'Number of used features is {X_train.shape[1]}')

## 5. Helper Functions

In [ ]:
def evaluate(model, name):
    """Print classification report and plot confusion matrix."""
    y_pred = model.predict(X_test)
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malware']))
    ax = sns.heatmap(confusion_matrix(y_pred, y_test), annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False)
    ax.set_xlabel('Predicted labels'); ax.set_ylabel('True labels')
    plt.title(name); plt.show()
    return y_pred

def plot_importance(model, title='Features importance'):
    """Plot feature importance bar chart."""
    imp = dict(zip(used_data.columns, model.feature_importances_))
    imp = dict(sorted(imp.items(), key=lambda x: x[1], reverse=True))
    plt.figure(figsize=(18, 28))
    sns.barplot(y=list(imp.keys()), x=list(imp.values()), palette='mako')
    plt.title(title); plt.show()

## 6. Random Forest Classifier

In [ ]:
rfc = RandomForestClassifier(n_estimators=100, random_state=0, oob_score=True, max_depth=16)
rfc.fit(X_train, y_train)
evaluate(rfc, 'Random Forest')
plot_importance(rfc)

## 7. Decision Tree Classifier

In [ ]:
dtc = DecisionTreeClassifier(random_state=0)
dtc.fit(X_train, y_train)
evaluate(dtc, 'Decision Tree')
plot_importance(dtc)

## 8. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=0)
lr.fit(X_train, y_train)
evaluate(lr, 'Logistic Regression')

## 9. Support Vector Machine (SVM)

In [ ]:
svm = SVC(random_state=0)
svm.fit(X_train, y_train)
evaluate(svm, 'SVM')

## 10. K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
evaluate(knn, 'KNN')

## 11. Model Comparison

In [ ]:
models = {'Random Forest': rfc, 'Decision Tree': dtc, 'Logistic Regression': lr, 'SVM': svm, 'KNN': knn}
acc = {name: m.score(X_test, y_test) for name, m in models.items()}
print(pd.DataFrame(acc.items(), columns=['Model', 'Accuracy']).sort_values('Accuracy', ascending=False).to_string(index=False))